Project - My Taste of Bihar


In [13]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [14]:
# Initialization

from pathlib import Path

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(Path.cwd() / ".env", override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")
    openai = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
    MODEL = "gemini-3.6-flash"

API key found and looks good so far!


In [15]:
system_message = system_message = """
You are a helpful assistant specializing exclusively in authentic Bihari cuisine.

You provide detailed information only about traditional Bihari food, including:
- Bihari dishes
- Ingredients
- Recipes
- Cooking methods
- Regional variations
- Food history and traditions
- Serving methods

Only provide information about authentic Bihari food.

If the user asks about food that is not Bihari cuisine, politely explain that you only assist with authentic Bihari cuisine.
"""

In [16]:
# Database setup

DB = "bihari_food.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS bihari_food (
            dish TEXT PRIMARY KEY,
            description TEXT,
            ingredients TEXT,
            preparation TEXT,
            region TEXT
        )
        """
    )
    conn.commit()

print("Database ready!")

Database ready!


In [17]:
def get_bihari_food_details(dish):
    print(f"DATABASE TOOL CALLED: Getting details for {dish}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT dish, description, ingredients, preparation, region
            FROM bihari_food
            WHERE LOWER(dish) = ?
            """,
            (dish.lower(),)
        )

        result = cursor.fetchone()

    if result:
        return f"""
Dish: {result[0]}
Description: {result[1]}
Ingredients: {result[2]}
Preparation: {result[3]}
Region: {result[4]}
"""

    return f"No authentic Bihari food information available for {dish}."

In [18]:
get_bihari_food_details("Litti Chokha")

DATABASE TOOL CALLED: Getting details for Litti Chokha


'\nDish: Litti Chokha\nDescription: A signature Bihari dish of wheat flour balls stuffed with sattu and served with mashed roasted vegetables.\nIngredients: Whole wheat flour, sattu, mustard oil, garlic, ginger, green chili, eggplant, potato, tomato, onion\nPreparation: Litti is baked over coal or in an oven until crisp. Chokha is made by roasting eggplant, potato, and tomato, then mashing with mustard oil and spices.\nRegion: All Bihar\n'

In [19]:
food_function = {
    "name": "get_bihari_food_details",
    "description": "Get detailed information about an authentic Bihari dish, including its ingredients, preparation method, description, and region.",
    "parameters": {
        "type": "object",
        "properties": {
            "dish": {
                "type": "string",
                "description": "The authentic Bihari dish for which the user wants information."
            }
        },
        "required": ["dish"],
        "additionalProperties": False
    }
}

tools = [
    {
        "type": "function",
        "function": food_function
    }
]

In [20]:
import json

def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_bihari_food_details":

            arguments = json.loads(
                tool_call.function.arguments
            )

            dish = arguments.get("dish")

            food_details = get_bihari_food_details(dish)

            responses.append({
                "role": "tool",
                "content": food_details,
                "tool_call_id": tool_call.id
            })

    return responses

In [21]:
def chat(message, history):

    history = [
        {
            "role": h["role"],
            "content": h["content"]
        }
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message

        responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
import gradio as gr

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/vikash/Documents/MyTaste/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vikash/Documents/MyTaste/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vikash/Documents/MyTaste/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vikash/Documents/MyTaste/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1688, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vikash/Documents/MyTaste/.venv/lib/python3.12/site-packages/gradio/utils.py", line 1081, 